# Databricks Lakehouse - Data Quality & Validation Audit (`04_data_quality`)

## Overview
This notebook performs a comprehensive data quality audit on raw telemetry records ingested into the **Bronze Layer** (`workspace.neuro_insight.bronze_telemetry`). It evaluates each record against predefined clinical bounds and structural validation rules, labeling each row as either `PASS` or `FAIL` to summarize overall data hygiene prior to Silver-layer ingestion.

---

### Step 1: Load Raw Bronze Data
Import PySpark functions and load telemetry records from the Bronze Delta table.

In [ ]:
from pyspark.sql import functions as F

# Read raw telemetry data from the Bronze table
bronze_df = spark.table("workspace.neuro_insight.bronze_telemetry")

print("Bronze records count:", bronze_df.count())

### Step 2: Evaluate Quality Rules & Tag Status
Apply conditional logic (`F.when`) to mark records as `FAIL` if they violate any of the following constraints[cite: 4]:
- **Missing Keys or Timestamps**: `patient_id`, `device_id`, or `event_timestamp` is null[cite: 4].
- **Out-of-Bound Tremor Score**: Outside the range $[0, 10]$[cite: 4].
- **Invalid Reaction Time or Gait Speed**: Value is $\le 0$[cite: 4].
- **Out-of-Bound Cognitive Score**: Outside the range $[0, 100]$[cite: 4].
- **Physiologically Unrealistic Heart Rate**: Outside the range $[30, 220]$ BPM[cite: 4].

Records passing all rules are tagged as `PASS`[cite: 4].

In [ ]:
# Evaluate quality rules and create the quality_status flag
quality_result = bronze_df.withColumn(
    "quality_status",
    F.when(
        F.col("patient_id").isNull()
        | F.col("device_id").isNull()
        | F.col("event_timestamp").isNull()
        | ~F.col("tremor_score").between(0, 10)
        | (F.col("reaction_time_ms") <= 0)
        | (F.col("gait_speed_mps") <= 0)
        | ~F.col("cognitive_score").between(0, 100)
        | ~F.col("heart_rate_bpm").between(30, 220),
        "FAIL"
    ).otherwise("PASS")
)

### Step 3: Aggregate & Summarize Quality Audit Results
Group data by `quality_status` to compute total passing vs. failing record counts[cite: 4].

In [ ]:
# Display summary table of passing and failing records
display(
    quality_result
    .groupBy("quality_status")
    .count()
)